# RQ5: Statistical Analysis — Subscription Tier Performance Comparison

**Research Question:** Do the three subscription tiers (Basic, Standard, Premium) produce statistically significantly different ROI, Revenue_Generated, and Units_Sold outcomes, and which tier demonstrates superior marketing campaign effectiveness after correcting for multiple comparisons?

**Task:** Statistical Hypothesis Testing  
**Groups:** Subscription_Tier (Basic / Standard / Premium)  
**Metrics:** ROI, Revenue_Generated, Units_Sold  
**Tests:** One-way ANOVA + Tukey HSD; Kruskal-Wallis + Mann-Whitney U + Bonferroni  
**Dataset:** Marketing and Product Performance Dataset (Kaggle)

In [80]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3', '#FF5722', '#4CAF50']
RANDOM_STATE = 42
OUTPUT_DIR = '/kaggle/working/'

def save_figure(fig, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, format='pdf', bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved figure: {path}')

def save_table(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f'Saved table:  {path}')

def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx-1)*x.std()**2 + (ny-1)*y.std()**2) / (nx+ny-2))
    return (x.mean() - y.mean()) / pooled_std if pooled_std > 0 else 0.0

def eta_squared(groups):
    all_data = np.concatenate(groups)
    grand_mean = all_data.mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total   = sum((x - grand_mean)**2 for x in all_data)
    return ss_between / ss_total if ss_total > 0 else 0.0

print('Imports OK')

Imports OK


## 1. Data Loading

In [81]:
input_dir = '/kaggle/input'
data_files = [
    os.path.join(root, f)
    for root, dirs, files in os.walk(input_dir)
    for f in files if f.endswith('.xlsx') or f.endswith('.xls') or f.endswith('.csv')
]
print('Found files:', data_files)
FILE_PATH = data_files[0]

df = pd.read_csv(FILE_PATH) if FILE_PATH.endswith('.csv') else pd.read_excel(FILE_PATH)
print(f'Shape: {df.shape}')

METRICS   = ['ROI', 'Revenue_Generated', 'Units_Sold']
TIER_COL  = 'Subscription_Tier'
TIERS     = sorted(df[TIER_COL].dropna().unique())
print(f'Subscription tiers: {TIERS}')
print(df.groupby(TIER_COL).size())

Found files: ['/kaggle/input/datasets/vanishjr/marketing-product-performance/marketing_and_product_performance.csv']
Shape: (10000, 17)
Subscription tiers: ['Basic', 'Premium', 'Standard']
Subscription_Tier
Basic       3416
Premium     3284
Standard    3300
dtype: int64


## 2. Descriptive Statistics

In [82]:
desc_rows = []
for tier in TIERS:
    sub = df[df[TIER_COL] == tier]
    row = {'Subscription_Tier': tier, 'Count': len(sub)}
    for m in METRICS:
        row[f'{m}_Mean']   = round(sub[m].mean(), 4)
        row[f'{m}_Std']    = round(sub[m].std(), 4)
        row[f'{m}_Median'] = round(sub[m].median(), 4)
        row[f'{m}_Min']    = round(sub[m].min(), 4)
        row[f'{m}_Max']    = round(sub[m].max(), 4)
    desc_rows.append(row)

desc_df = pd.DataFrame(desc_rows)
save_table(desc_df, 'rq5_descriptive_stats.csv')
desc_df

Saved table:  /kaggle/working/rq5_descriptive_stats.csv


,Subscription_Tier,Count,ROI_Mean,ROI_Std,ROI_Median,ROI_Min,ROI_Max,Revenue_Generated_Mean,Revenue_Generated_Std,Revenue_Generated_Median,Revenue_Generated_Min,Revenue_Generated_Max,Units_Sold_Mean,Units_Sold_Std,Units_Sold_Median,Units_Sold_Min,Units_Sold_Max
0,Basic,3416,2.7908,1.2909,2.85,0.5,5.0,50157.9182,28425.9808,49564.080,1020.84,99995.06,101.7930,57.2814,103.0,1,199
1,Premium,3284,2.7394,1.2991,2.69,0.5,5.0,50093.1398,28566.1351,50360.955,1049.04,99999.47,99.9534,58.0532,100.0,1,199
2,Standard,3300,2.7376,1.3006,2.73,0.5,5.0,49860.8958,28656.6067,48727.495,1002.08,99958.93,100.2800,55.8634,99.0,1,199


## 3. Normality & Variance Homogeneity Tests

In [83]:
norm_rows = []
for metric in METRICS:
    for tier in TIERS:
        grp = df[df[TIER_COL] == tier][metric].dropna().values
        sample = grp[:5000]  # Shapiro-Wilk limit
        w, p = stats.shapiro(sample)
        norm_rows.append({'Metric': metric, 'Tier': tier,
                          'Shapiro_W': round(w, 4), 'Shapiro_p': round(p, 4),
                          'Normal': p > 0.05})

norm_df = pd.DataFrame(norm_rows)
print('Normality tests (Shapiro-Wilk):')
print(norm_df.to_string(index=False))
print()

for metric in METRICS:
    groups = [df[df[TIER_COL] == t][metric].dropna().values for t in TIERS]
    stat, p = stats.levene(*groups)
    print(f'Levene test for {metric}: stat={stat:.4f}  p={p:.4f}  Equal variance: {p > 0.05}')

Normality tests (Shapiro-Wilk):
           Metric     Tier  Shapiro_W  Shapiro_p  Normal
              ROI    Basic     0.9561        0.0   False
              ROI  Premium     0.9546        0.0   False
              ROI Standard     0.9534        0.0   False
Revenue_Generated    Basic     0.9541        0.0   False
Revenue_Generated  Premium     0.9537        0.0   False
Revenue_Generated Standard     0.9530        0.0   False
       Units_Sold    Basic     0.9562        0.0   False
       Units_Sold  Premium     0.9515        0.0   False
       Units_Sold Standard     0.9615        0.0   False

Levene test for ROI: stat=0.8550  p=0.4253  Equal variance: True
Levene test for Revenue_Generated: stat=0.2477  p=0.7806  Equal variance: True
Levene test for Units_Sold: stat=6.6222  p=0.0013  Equal variance: False


## 4. One-Way ANOVA + Tukey HSD

In [84]:
anova_results = []
for metric in METRICS:
    groups = [df[df[TIER_COL] == t][metric].dropna().values for t in TIERS]
    F, p = stats.f_oneway(*groups)
    eta2 = eta_squared(groups)
    anova_results.append({'Metric': metric, 'F_stat': round(F, 4), 'p_value': round(p, 6),
                          'Significant': p < 0.05, 'Eta_squared': round(eta2, 4)})
    print(f'ANOVA {metric}: F={F:.4f}  p={p:.4f}  η²={eta2:.4f}')

    if p < 0.05:
        tukey = pairwise_tukeyhsd(
            endog=df[metric].dropna(),
            groups=df.loc[df[metric].notna(), TIER_COL]
        )
        print(tukey.summary())

ANOVA ROI: F=1.8313  p=0.1603  η²=0.0004
ANOVA Revenue_Generated: F=0.0998  p=0.9051  η²=0.0000
ANOVA Units_Sold: F=0.9966  p=0.3692  η²=0.0002


## 5. Kruskal-Wallis + Pairwise Mann-Whitney U (Bonferroni)

In [85]:
pairwise_rows = []
tier_pairs = list(combinations(TIERS, 2))
n_comparisons = len(tier_pairs)

for metric in METRICS:
    groups = {t: df[df[TIER_COL] == t][metric].dropna().values for t in TIERS}
    H, p_kw = stats.kruskal(*groups.values())
    n = sum(len(v) for v in groups.values())
    k = len(TIERS)
    eps2 = (H - k + 1) / (n - k)
    print(f'Kruskal-Wallis {metric}: H={H:.4f}  p={p_kw:.4f}  ε²={eps2:.4f}')

    for (t1, t2) in tier_pairs:
        u, p_mw = stats.mannwhitneyu(groups[t1], groups[t2], alternative='two-sided')
        p_adj = min(p_mw * n_comparisons, 1.0)  # Bonferroni
        d = cohens_d(pd.Series(groups[t1]), pd.Series(groups[t2]))
        pairwise_rows.append({
            'Metric': metric, 'Tier_A': t1, 'Tier_B': t2,
            'U_stat': round(u, 2), 'p_raw': round(p_mw, 6),
            'p_bonferroni': round(p_adj, 6),
            'Significant': p_adj < 0.05,
            'Cohens_d': round(d, 4)
        })

pairwise_df = pd.DataFrame(pairwise_rows)
save_table(pairwise_df, 'rq5_posthoc_pairwise_tests.csv')
print(pairwise_df[pairwise_df['Significant']].to_string(index=False))

Kruskal-Wallis ROI: H=3.6618  p=0.1603  ε²=0.0002
Kruskal-Wallis Revenue_Generated: H=0.2069  p=0.9017  ε²=-0.0002
Kruskal-Wallis Units_Sold: H=1.9949  p=0.3688  ε²=-0.0000
Saved table:  /kaggle/working/rq5_posthoc_pairwise_tests.csv
Empty DataFrame
Columns: [Metric, Tier_A, Tier_B, U_stat, p_raw, p_bonferroni, Significant, Cohens_d]
Index: []


## 6. Publication-Ready Figures

In [86]:
# ── Violin Plots (3 metrics × 3 tiers) ──────────────────────────────────────
metric_labels = {'ROI': 'ROI', 'Revenue_Generated': 'Revenue Generated (USD)', 'Units_Sold': 'Units Sold'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, metric in zip(axes, METRICS):
    sns.violinplot(data=df, x=TIER_COL, y=metric, ax=ax, palette=PALETTE,
                   order=TIERS, inner='box', cut=0)
    sns.stripplot(data=df.sample(min(500, len(df)), random_state=RANDOM_STATE),
                  x=TIER_COL, y=metric, ax=ax, color='black', alpha=0.2,
                  size=2, order=TIERS, jitter=True)
    ax.set_title(f'{metric_labels[metric]}\nby Subscription Tier', fontsize=11, fontweight='bold')
    ax.set_xlabel('Subscription Tier')
    ax.set_ylabel(metric_labels[metric])

fig.suptitle('RQ5 — Campaign Performance by Subscription Tier', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'rq5_violin_plots.pdf')
plt.show()

Saved figure: /kaggle/working/rq5_violin_plots.pdf


In [87]:
# ── Post-hoc p-value Heatmaps ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, METRICS):
    sub_df = pairwise_df[pairwise_df['Metric'] == metric].copy()
    # Build symmetric p-value matrix
    p_matrix = pd.DataFrame(1.0, index=TIERS, columns=TIERS)
    for _, row in sub_df.iterrows():
        p_matrix.loc[row['Tier_A'], row['Tier_B']] = row['p_bonferroni']
        p_matrix.loc[row['Tier_B'], row['Tier_A']] = row['p_bonferroni']

    sns.heatmap(p_matrix, annot=True, fmt='.4f', cmap='RdYlGn_r',
                vmin=0, vmax=0.1, ax=ax, linewidths=0.5, square=True,
                cbar_kws={'label': 'p-value (Bonferroni)'})
    ax.set_title(f'{metric}\nPairwise p-values', fontsize=11, fontweight='bold')

fig.suptitle('RQ5 — Post-hoc Pairwise Test p-values (Bonferroni)', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'rq5_posthoc_pvalue_heatmap.pdf')
plt.show()

Saved figure: /kaggle/working/rq5_posthoc_pvalue_heatmap.pdf


## 7. Conclusions

In [88]:
print('=' * 60)
print('RQ5 CONCLUSION')
print('=' * 60)
for row in anova_results:
    sig = 'SIGNIFICANT' if row['Significant'] else 'not significant'
    print(f"{row['Metric']}: F={row['F_stat']}  p={row['p_value']}  η²={row['Eta_squared']}  [{sig}]")
print()
sig_pairs = pairwise_df[pairwise_df['Significant']]
if len(sig_pairs) > 0:
    print('Significant pairwise differences (Bonferroni-corrected):')
    for _, row in sig_pairs.iterrows():
        print(f"  {row['Metric']}: {row['Tier_A']} vs {row['Tier_B']}  d={row['Cohens_d']}")
else:
    print('No significant pairwise differences after Bonferroni correction.')
print()
print('Outputs saved:')
for f in ['rq5_violin_plots.pdf','rq5_posthoc_pvalue_heatmap.pdf',
          'rq5_descriptive_stats.csv','rq5_posthoc_pairwise_tests.csv']:
    print(f'  {f}')

RQ5 CONCLUSION
ROI: F=1.8313  p=0.160266  η²=0.0004  [not significant]
Revenue_Generated: F=0.0998  p=0.905054  η²=0.0  [not significant]
Units_Sold: F=0.9966  p=0.369172  η²=0.0002  [not significant]

No significant pairwise differences after Bonferroni correction.

Outputs saved:
  rq5_violin_plots.pdf
  rq5_posthoc_pvalue_heatmap.pdf
  rq5_descriptive_stats.csv
  rq5_posthoc_pairwise_tests.csv
